# 05 · Rollout 数据与一次更新调试

## 学习目标

观察真实采样 batch 的 shape、detach 状态，以及首次更新前 ratio=1 的条件。

本 Notebook 直接运行已补全的 exercise 实现；修改 `.py` 后重新运行相关 Cell。

## 环境准备

先按 [使用指南](README.md) 安装 `.[dev,notebook]`，选择项目 Kernel。

In [1]:
from pathlib import Path
import os
import sys
import torch
import numpy as np

# 从仓库根目录或 notebooks/ 打开都可；不要依赖某台机器的绝对路径。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').exists()
                     and (p / 'src/ppo_pendulum').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目目录启动 JupyterLab')
os.chdir(PROJECT_ROOT)
%load_ext autoreload
%autoreload 2
torch.manual_seed(7)
np.random.seed(7)
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT.name)

Python: /Users/matt/Documents/Codex/2026-11-15/referenced-chatgpt-conversation-this-is-an-2/work/.venv/bin/python
Project: ppo-inverted-pendulum


## 1. 收集一个短 rollout

Trainer 在 CPU 运行，显式关闭环境。batch 保存到内核内存后仍可查看。

In [2]:
from dataclasses import fields
from ppo_pendulum.config import PPOConfig
from ppo_pendulum.exercises.trainer import ExercisePPOTrainer
config = PPOConfig(total_timesteps=64, rollout_steps=64,
                   minibatch_size=32, update_epochs=2, seed=7,
                   max_episode_steps=5)
trainer = ExercisePPOTrainer(config)
try:
    batch = trainer.collect_rollout(64)
    for field in fields(batch):
        tensor = getattr(batch, field.name)
        print(f'{field.name:16s} shape={str(tuple(tensor.shape)):10s} requires_grad={tensor.requires_grad}')
    print('completed episode lengths:', trainer.completed_lengths[:8])
    assert batch.observations.shape == (64, 4)
    assert batch.raw_actions.shape == (64, 1)
    assert all(not getattr(batch, f.name).requires_grad for f in fields(batch))
    with torch.no_grad():
        new_log_prob, _, _ = trainer.model.evaluate_actions(batch.observations, batch.raw_actions)
        ratios_before = (new_log_prob - batch.old_log_probs).exp()
    torch.testing.assert_close(ratios_before, torch.ones(64))
    print('ratio before update:', ratios_before.min().item(), ratios_before.max().item())
finally:
    trainer.close()

observations     shape=(64, 4)    requires_grad=False
raw_actions      shape=(64, 1)    requires_grad=False
old_log_probs    shape=(64,)      requires_grad=False
advantages       shape=(64,)      requires_grad=False
returns          shape=(64,)      requires_grad=False
values           shape=(64,)      requires_grad=False
completed episode lengths: [5, 5, 5, 5, 5, 5, 5, 5]
ratio before update: 0.9999998211860657 1.0000001192092896


## 2. 单独执行一次 update

更新仅依赖 batch 和模型，不再访问环境。可在下面 update 行前设置 `breakpoint()`，进入源码逐行查看。

In [3]:
old_log_probs_copy = batch.old_log_probs.clone()
stats = trainer.update(batch)
print({key: round(value, 6) for key, value in stats.items()})
torch.testing.assert_close(batch.old_log_probs, old_log_probs_copy)
assert all(np.isfinite(value) for value in stats.values())
with torch.no_grad():
    new_log_prob, _, _ = trainer.model.evaluate_actions(batch.observations, batch.raw_actions)
    ratios_after = (new_log_prob - batch.old_log_probs).exp()
print('ratio after update range:', ratios_after.min().item(), ratios_after.max().item())

{'actor_loss': -0.005894, 'critic_loss': 4.043945, 'entropy': 0.918489, 'approx_kl': 0.000626, 'clip_fraction': 0.0}
ratio after update range: 0.8013858795166016 1.16360342502594


## 检查完整 exercise 管线

上面的 `ExercisePPOTrainer` 已调用 `exercises/rollout.py` 和
`exercises/update.py`。你可以在两个文件中设置断点，再重新运行本页：

- rollout 必须保存 raw action，并在 reset 前计算 next value；
- GAE 完成后才允许打乱 mini-batch；
- update 前 ratio 应为 1，update 后可偏离 1；
- old log probability、advantage 和 target 都不保留梯度图。

## 下一步

打开 `06_end_to_end_training.ipynb`，执行完整短训练并读取输出。

公式对照：[PPO_FORMULAS.md](../docs/PPO_FORMULAS.md)。